In [1]:
import cvxpy as cp
import numpy as np
import random

In [2]:
def _compress(row):
    """Slide non-zero tiles to the left."""
    non_zero = row[row != 0]
    return np.concatenate([non_zero, np.zeros(len(row) - len(non_zero), dtype=row.dtype)])

def _merge(row):
    """Merge equal adjacent tiles (leftward), once per pair."""
    row = row.copy()
    score = 0
    for i in range(len(row) - 1):
        if row[i] != 0 and row[i] == row[i + 1]:
            row[i] *= 2
            row[i + 1] = 0
            score += row[i]
    return row, score

def _move_left(board):
    """Apply a LEFT move to the board."""
    new_board = np.zeros_like(board)
    total_score = 0
    for i in range(board.shape[0]):
        row = board[i, :]
        row = _compress(row)
        row, score = _merge(row)
        row = _compress(row)
        new_board[i, :] = row
        total_score += score
    return new_board, total_score

def _move_right(board):
    """Apply a RIGHT move to the board."""
    flipped = np.fliplr(board)
    moved, score = _move_left(flipped)
    return np.fliplr(moved), score

def _move_up(board):
    """Apply an UP move to the board."""
    rotated = board.T
    moved, score = _move_left(rotated)
    return moved.T, score

def _move_down(board):
    """Apply a DOWN move to the board."""
    rotated = board.T
    moved, score = _move_right(rotated)
    return moved.T, score

def apply_move(board, direction):
    """
    direction: 0=LEFT, 1=RIGHT, 2=UP, 3=DOWN
    returns: (new_board, merge_score)
    """
    if direction == 0:
        return _move_left(board)
    elif direction == 1:
        return _move_right(board)
    elif direction == 2:
        return _move_up(board)
    elif direction == 3:
        return _move_down(board)
    else:
        raise ValueError("Invalid direction")


In [3]:
I, J = 4, 4
D = 4  # 0=LEFT,1=RIGHT,2=UP,3=DOWN
M = 2048.0

# Example board
b = np.array([
    [2, 0, 0, 0],
    [2, 2, 0, 0],
    [4, 0, 0, 0],
    [0, 0, 0, 0]
], dtype=float)

# --- Decision variables ---

# Move choice
y = cp.Variable(D, boolean=True)          # y[d] in {0,1}

# Post-move board
x = cp.Variable((I, J))                   # after move

# Example: LEFT-merge "score" variables (toy, 1 per row)
# In a full model you'd have s_L[i,k] for k=1..3, etc.
s_L = cp.Variable(I)                      # merge score per row for LEFT
s_R = cp.Variable(I)                      # for RIGHT
s_U = cp.Variable(I)                      # for UP
s_D = cp.Variable(I)                      # for DOWN

constraints = []

# --- Move selection ---
constraints += [cp.sum(y) == 1]

# --- Toy gating: only allow LEFT merges if y[0] = 1, etc. ---

# Example: bound merge scores by big-M and move choice
for i in range(I):
    # LEFT
    constraints += [s_L[i] >= 0,
                    s_L[i] <= M * y[0]]
    # RIGHT
    constraints += [s_R[i] >= 0,
                    s_R[i] <= M * y[1]]
    # UP
    constraints += [s_U[i] >= 0,
                    s_U[i] <= M * y[2]]
    # DOWN
    constraints += [s_D[i] >= 0,
                    s_D[i] <= M * y[3]]

# --- Toy board constraints (replace with full 2048 physics) ---
# For now, just force x == b so the model is feasible and linear.
constraints += [x >= 0, x <= M]

# --- Score: purely linear, no products ---
score = cp.sum(s_L) + cp.sum(s_R) + cp.sum(s_U) + cp.sum(s_D)

objective = cp.Maximize(score)

prob = cp.Problem(objective, constraints)
prob.solve(solver=cp.HIGHS)

print("Status:", prob.status)
print("Move choice y:", y.value)
print("Score:", score.value)


Status: optimal
Move choice y: [0. 0. 0. 1.]
Score: 8192.0


In [4]:
y_val = y.value  # shape (4,)
move_idx = int(np.argmax(y_val))  # 0=LEFT,1=RIGHT,2=UP,3=DOWN
x_board, merge_score = apply_move(b, move_idx)
x_board,merge_score

(array([[0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [4., 0., 0., 0.],
        [4., 2., 0., 0.]]),
 np.float64(4.0))

In [5]:
# -----------------------------
# 2048 engine (true physics)
# -----------------------------
def _compress(row):
    non_zero = row[row != 0]
    return np.concatenate([non_zero, np.zeros(len(row) - len(non_zero), dtype=row.dtype)])

def _merge(row):
    row = row.copy()
    score = 0
    for i in range(len(row) - 1):
        if row[i] != 0 and row[i] == row[i + 1]:
            row[i] *= 2
            row[i + 1] = 0
            score += row[i]          # score = value of merged tile
    return row, score

def _move_left(board):
    new_board = np.zeros_like(board)
    total_score = 0
    for i in range(board.shape[0]):
        row = board[i, :]
        row = _compress(row)
        row, score = _merge(row)
        row = _compress(row)
        new_board[i, :] = row
        total_score += score
    return new_board, total_score

def _move_right(board):
    flipped = np.fliplr(board)
    moved, score = _move_left(flipped)
    return np.fliplr(moved), score

def _move_up(board):
    rotated = board.T
    moved, score = _move_left(rotated)
    return moved.T, score

def _move_down(board):
    rotated = board.T
    moved, score = _move_right(rotated)
    return moved.T, score

def apply_move(board, direction):
    # direction: 0=LEFT,1=RIGHT,2=UP,3=DOWN
    if direction == 0:
        return _move_left(board)
    elif direction == 1:
        return _move_right(board)
    elif direction == 2:
        return _move_up(board)
    elif direction == 3:
        return _move_down(board)
    else:
        raise ValueError("Invalid direction")

# -----------------------------
# Initial board
# -----------------------------
b = np.array([
    [2, 0, 0, 0],
    [2, 2, 0, 0],
    [4, 0, 0, 0],
    [0, 0, 0, 0]
], dtype=float)

# -----------------------------
# Precompute physics per move
# -----------------------------
D = 4  # 0=LEFT,1=RIGHT,2=UP,3=DOWN
boards_after = []
scores = []

for d in range(D):
    new_board, merge_score = apply_move(b, d)
    boards_after.append(new_board)
    scores.append(merge_score)

scores = np.array(scores, dtype=float)

print("Scores per direction [L,R,U,D]:", scores)

# -----------------------------
# MILP: choose optimal direction
# -----------------------------
y = cp.Variable(D, boolean=True)   # move choice
constraints = [cp.sum(y) == 1]     # exactly one move

# Objective: true 2048 score (linear, since scores are constants)
objective = cp.Maximize(scores @ y)

prob = cp.Problem(objective, constraints)
prob.solve(solver=cp.HIGHS)

print("Status:", prob.status)
print("y:", y.value)

best_dir = int(np.argmax(y.value))
print("Best direction index:", best_dir, "(0=L,1=R,2=U,3=D)")
print("Best score:", scores[best_dir])
print("Resulting board:\n", boards_after[best_dir])


Scores per direction [L,R,U,D]: [4. 4. 4. 4.]
Status: optimal
y: [0. 0. 0. 1.]
Best direction index: 3 (0=L,1=R,2=U,3=D)
Best score: 4.0
Resulting board:
 [[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [4. 0. 0. 0.]
 [4. 2. 0. 0.]]


In [6]:
# ---------------------------------------------------------
# 2048 ENGINE (same as before)
# ---------------------------------------------------------
def _compress(row):
    non_zero = row[row != 0]
    return np.concatenate([non_zero, np.zeros(len(row) - len(non_zero), dtype=row.dtype)])

def _merge(row):
    row = row.copy()
    score = 0
    for i in range(len(row) - 1):
        if row[i] != 0 and row[i] == row[i + 1]:
            row[i] *= 2
            row[i + 1] = 0
            score += row[i]
    return row, score

def _move_left(board):
    new_board = np.zeros_like(board)
    total_score = 0
    for i in range(board.shape[0]):
        row = board[i, :]
        row = _compress(row)
        row, score = _merge(row)
        row = _compress(row)
        new_board[i, :] = row
        total_score += score
    return new_board, total_score

def _move_right(board):
    flipped = np.fliplr(board)
    moved, score = _move_left(flipped)
    return np.fliplr(moved), score

def _move_up(board):
    rotated = board.T
    moved, score = _move_left(rotated)
    return moved.T, score

def _move_down(board):
    rotated = board.T
    moved, score = _move_right(rotated)
    return moved.T, score

def apply_move(board, direction):
    if direction == 0: return _move_left(board)
    if direction == 1: return _move_right(board)
    if direction == 2: return _move_up(board)
    if direction == 3: return _move_down(board)
    raise ValueError("Invalid direction")

# ---------------------------------------------------------
# SPAWN LOGIC
# ---------------------------------------------------------
def spawn_tile(board):
    empty = list(zip(*np.where(board == 0)))
    if not empty:
        return board
    i, j = random.choice(empty)
    board[i, j] = 2 if random.random() < 0.9 else 4
    return board

# ---------------------------------------------------------
# MIP GREEDY MOVE SELECTOR
# ---------------------------------------------------------
def mip_greedy_move(board):
    D = 4
    boards_after = []
    scores = []

    # Precompute physics for each direction
    for d in range(D):
        new_board, merge_score = apply_move(board, d)
        boards_after.append(new_board)
        scores.append(merge_score)

    scores = np.array(scores, dtype=float)

    # If all moves are illegal, return None
    if all(np.array_equal(boards_after[d], board) for d in range(4)):
        return None, 0

    # MILP: choose best direction
    y = cp.Variable(D, boolean=True)
    constraints = [cp.sum(y) == 1]

    objective = cp.Maximize(scores @ y)
    prob = cp.Problem(objective, constraints)
    prob.solve(solver=cp.HIGHS)

    best_dir = int(np.argmax(y.value))
    return best_dir, scores[best_dir]

# ---------------------------------------------------------
# FULL GAME SIMULATION
# ---------------------------------------------------------
def run_mip_greedy_game(seed=None):
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)

    board = np.zeros((4,4), dtype=int)
    board = spawn_tile(board)
    board = spawn_tile(board)

    total_score = 0
    moves = 0
    
    while True:
        d, score = mip_greedy_move(board)
        if d is None:
            break

        board, gained = apply_move(board, d)
        total_score += gained
        moves += 1

        board = spawn_tile(board)

    return {
        "score": total_score,
        "max_tile": int(board.max()),
        "moves": moves,
        "final_board": board
    }

# ---------------------------------------------------------
# RUN MANY GAMES
# ---------------------------------------------------------
def simulate_mip_greedy(n_games=20):
    results = [run_mip_greedy_game() for _ in range(n_games)]
    scores = [r["score"] for r in results]
    max_tiles = [r["max_tile"] for r in results]

    print(f"Games played: {n_games}")
    print(f"Average score: {np.mean(scores):.1f}")
    print(f"Median score:  {np.median(scores):.1f}")
    print(f"Max score:     {np.max(scores)}")
    print(f"Average max tile: {np.mean(max_tiles):.1f}")
    print("Max tile distribution:")
    for t in sorted(set(max_tiles)):
        print(f"  {t}: {max_tiles.count(t)} games")

    return results

# Example: run 20 greedy-MIP games
results = simulate_mip_greedy(1)
results

Games played: 1
Average score: 1600.0
Median score:  1600.0
Max score:     1600
Average max tile: 128.0
Max tile distribution:
  128: 1 games


[{'score': np.int64(1600),
  'max_tile': 128,
  'moves': 168,
  'final_board': array([[ 16,   8,   2,   4],
         [ 32,  16,   4,   2],
         [  2,  32,  16,   4],
         [128,  64,  32,   8]])}]

# Greedy One‑Move Optimizer for 2048 Using a MILP Selector

This model implements a **greedy one‑move optimizer** for the game *2048*.  
The key idea is to separate responsibilities:

1. **Game physics and scoring**  
   - A Python engine computes the *true* result of each possible move  
     (LEFT, RIGHT, UP, DOWN).  
   - For each direction, we obtain:
     - the resulting board  
     - the exact merge score (sum of merged tile values), following the real 2048 rules.

2. **Optimization**  
   - A small MILP chooses the direction that maximizes the immediate merge score.  
   - Because the merge scores are constants (precomputed by the engine),  
     the MILP is fully linear and DCP‑compliant.

This produces a **greedy one‑step policy**:  
> Choose the move that yields the highest immediate score.

---

## 1. True 2048 Physics

For each direction \( d \in \{L, R, U, D\} \), we apply the real 2048 rules:

- **Compress** tiles toward the move direction  
- **Merge** equal adjacent tiles (once per pair)  
- **Compress again**  
- **Score** = sum of merged tile values

This ensures the optimization uses **exact game behavior**, not an approximation.

---

## 2. Precomputing Move Outcomes

For the current board \( b \), we compute:

- \( \text{board\_after}[d] \): the board after applying move \( d \)  
- \( \text{score}[d] \): the merge score for move \( d \)

These values are constants from the MILP’s perspective.

---

## 3. MILP Formulation

### Decision variable
We introduce a binary vector:



\$
y_d \in \{0,1\}, \quad d \in \{L,R,U,D\}
\$



with the constraint:



\$
\sum_d y_d = 1
\$



This forces the solver to choose **exactly one** direction.

### Objective
Because the merge scores are constants, the objective is linear:



\$
\max \sum_{d} y_d \cdot \text{score}[d]
\$



This selects the direction with the highest immediate reward.

---

## 4. Interpretation

This model is a **greedy one‑move optimizer**:

- It does **not** consider future tile spawns  
- It does **not** plan multiple steps ahead  
- It does **not** evaluate long‑term board quality  

It simply chooses the move that yields the **largest immediate merge score**,  
exactly matching the scoring rules of 2048.

This is the foundation for more advanced planners such as:

- multi‑step lookahead  
- scenario‑based stochastic optimization  
- MPC (model predictive control)  
- hybrid MIP + value function methods

---
